# Tag 3: Umzug auf eine SQLite-Datenbank

**Tagesziel:** Wir ersetzen JSON durch eine echte relationale Datenbank und bauen eine **Bestenliste der staerksten Helden**. Zusaetzlich: professionelles Tooling (DB Browser), Security (SQL-Injection), Context Manager.

Aufgaben und Code stehen separat in `aufgaben.py`, `aufgaben_loesung.py`, `dungeon.py` und `dungeon_loesung.py` - dieses Notebook enthaelt ausschliesslich Theorie.

---
## 3.1 SQLite Grundlagen

**Definition:** SQLite ist eine dateibasierte, in Python fest eingebaute Datenbank. `connect()` oeffnet die Datenbankdatei, ein `cursor` fuehrt SQL-Befehle aus.

**Warum wichtig:** Eine JSON-Datei speichert immer nur einen Zustand auf einmal. Eine Datenbank kann beliebig viele Zeilen (z.B. jeden einzelnen Kampf) dauerhaft und durchsuchbar speichern.

**Syntax:**
```python
conn = sqlite3.connect("datei.db")
cursor = conn.cursor()
cursor.execute("CREATE TABLE IF NOT EXISTS ... (...)")
conn.commit()
conn.close()
```

**Tooling:** "DB Browser for SQLite" zum Oeffnen und Inspizieren der `.db`-Datei - installiert es vor dem Kurstag, falls noch nicht vorhanden.

**Merksatz:** *"IF NOT EXISTS macht CREATE TABLE gefahrlos wiederholbar - die Funktion kann bei jedem Programmstart erneut aufgerufen werden."*

---
## 3.2 INSERT, Security & Context Manager

**Definition:** `INSERT` fuegt eine neue Zeile ein. `?`-Platzhalter uebergeben Werte sicher getrennt vom SQL-Befehl. Der Context Manager (`with sqlite3.connect(...)`) uebernimmt `commit()` automatisch.

**Warum wichtig - Sicherheit:** Werden Eingaben direkt per f-String in den SQL-Befehl eingesetzt, kann eine boesartige Eingabe wie `Aria' OR '1'='1` den Befehl umschreiben und z.B. alle Zeilen loeschen (**SQL-Injection**). `?`-Platzhalter verhindern das zuverlaessig.

**Gefaehrliche Variante (nur zur Veranschaulichung, niemals verwenden):**
```python
eingabe = input("Heldenname zum Loeschen: ")
cursor.execute(f"DELETE FROM helden WHERE name = '{eingabe}'")  # GEFAEHRLICH
```
Eingabe `Aria' OR '1'='1` wuerde ALLE Zeilen loeschen.

**Sichere Syntax:**
```python
with sqlite3.connect("datei.db") as conn:
    cursor = conn.cursor()
    cursor.execute("INSERT INTO tabelle (spalte1, spalte2) VALUES (?, ?)", (wert1, wert2))
```

**Warum der Context Manager wichtig ist:** Ohne `with` muesste man `conn.commit()` und `conn.close()` manuell aufrufen - vergisst man das (oder tritt vorher ein Fehler auf), gehen Aenderungen verloren oder die Datei bleibt gesperrt. `with` erledigt das automatisch, selbst bei einem Fehler im Block.

**Merksatz:** *"Werte gehoeren immer als Tupel hinter das SQL-Statement, nie direkt hineingeschrieben - `?` ist Pflicht, kein Stil."*

---
## 3.3 SELECT, ORDER BY, LIMIT

**Definition:** `SELECT` liest Zeilen aus. `ORDER BY spalte ASC/DESC` sortiert, `LIMIT n` begrenzt die Anzahl. `fetchone()` liefert eine Zeile, `fetchall()` alle.

**Warum wichtig:** Fuer eine Bestenliste soll die Datenbank selbst sortieren und begrenzen, statt dass wir alle Daten laden und in Python nachbauen, was SQL bereits kann.

**Syntax:**
```python
cursor.execute("SELECT spalte FROM tabelle ORDER BY spalte DESC LIMIT n")
ergebnisse = cursor.fetchall()
```

**ASC vs. DESC:** `ASC` (aufsteigend, Standard) eignet sich z.B. fuer "wenigste Versuche zuerst", `DESC` (absteigend) fuer "hoechstes Level zuerst" - je nachdem, was in der jeweiligen Bestenliste als "besser" gilt.

**Haeufiger Fehler:** `fetchone()` liefert `None`, wenn die Abfrage kein Ergebnis findet (z.B. leere Tabelle). Wird das Ergebnis ungeprueft entpackt (`name, score = cursor.fetchone()`), fuehrt das zu einem `TypeError`, da `None` nicht entpackt werden kann.

**Merksatz:** *"fetchone() kann None liefern (Tabelle leer) - immer pruefen, bevor man das Ergebnis entpackt."*

---
## Zusammenspiel im Praxisprojekt

Im heutigen Praxisteil (`dungeon.py` / `dungeon_loesung.py`) verbinden sich alle drei Konzepte:

- `setup_db()` nutzt **CREATE TABLE IF NOT EXISTS** (3.1), um die Tabelle gefahrlos bei jedem Start anzulegen
- `speichere_held_db()` nutzt **INSERT mit `?`-Platzhaltern und Context Manager** (3.2)
- `get_top_5_helden()` nutzt **SELECT mit ORDER BY und LIMIT** (3.3), um eine echte Bestenliste zu bauen

**Tagesabschluss:** Jeder abgeschlossene Kampf landet jetzt als neue Zeile in der Datenbank - die Historie bleibt erhalten, nicht nur der letzte Wert. Datenbank-Code, Kampflogik und Konsolen-I/O liegen aber noch ungeordnet in einer Datei. **Ausblick:** Morgen Software-Architektur - Aufteilung in Module.